# import libraries

In [32]:
import pandas as pd 
import numpy as np 

# load point by point data

In [33]:
path_data = r'C:\Users\ASMA\Desktop\code\Tennis_project\data\intermediate_data\pbp.parquet'
pbp_data = pd.read_parquet (path_data)
pbp_data.head()

,match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
0,12041772,3,1,0,0,1,0,5,1,11,9,1,1
1,12041772,3,1,1,0,2,0,5,6,11,9,1,1
2,12041772,3,1,2,0,3,0,5,6,11,9,1,1
3,12041772,3,1,3,1,3,0,6,5,11,9,1,1
4,12041772,3,1,4,1,4,0,5,1,11,9,1,1


# Removing duplicate rows

In [34]:
rows_before = pbp_data.shape[0]
pbp_duplicated = pbp_data.duplicated()
duplicate_count = pbp_duplicated.sum()
print("Rows before remove duplicating:", rows_before)
print("Exact duplicate rows:", duplicate_count) 

pbp_data = pbp_data.drop_duplicates() 
print("Rows after remove duplicating:", pbp_data.shape[0])
print("Remaining exact duplicates:", pbp_data.duplicated().sum())

Rows before remove duplicating: 2549369
Exact duplicate rows: 1294625
Rows after remove duplicating: 1254744
Remaining exact duplicates: 0


# Determining the game type (normal / tie-break)

In [35]:
def detect_game_type(home_point, away_point):
    
    normal_scores = {'0', '15', '30', '40', 'A'}
    if home_point in normal_scores and away_point in normal_scores:
        return 'normal_game'
    else:
        return 'tie_break'
    
    
pbp_data['game_type'] = pbp_data.apply(
    lambda row: detect_game_type(row['home_point'], row['away_point']),
    axis=1
)

pbp_data['game_type'].value_counts()

game_type
normal_game    1215571
tie_break        39173
Name: count, dtype: int64

# Checking for consistent game type across games

In [36]:
game_type_check = (
    pbp_data
    .groupby(['match_id', 'set_id', 'game_id'])['game_type']
    .nunique()
)

print (game_type_check.value_counts())
print (game_type_check[game_type_check > 1])

game_type
1    228224
2         1
Name: count, dtype: int64
match_id  set_id  game_id
12041957  3       1          2
Name: game_type, dtype: int64


# Identifying and correcting games that simultaneously feature both the Normal and Tie-Break models

In [37]:
mask = (
    (pbp_data['match_id'] == 12041957) &
    (pbp_data['set_id'] == 3) &
    (pbp_data['game_id'] == 1)
)

pbp_data.loc[mask, 'game_type'] = 'tie_break'

game_type_check = (
    pbp_data
    .groupby(['match_id', 'set_id', 'game_id'])['game_type']
    .nunique()
)
print (game_type_check[game_type_check > 1])


Series([], Name: game_type, dtype: int64)


# Create tie break game dataframe

In [38]:
tie_break_data = pbp_data[(pbp_data['game_type'] == 'tie_break')].copy()
tie_break_data['home_point'] = pd.to_numeric(tie_break_data['home_point'])
tie_break_data['away_point']= pd.to_numeric(tie_break_data['away_point'])


# validate tie break game data

In [39]:
def validate_tie_break(group):
    group = group.sort_values('point_id').copy()

    # 1. Scores must not decrease
    home_diff = group['home_point'].diff()
    away_diff = group['away_point'].diff()

    if (home_diff < 0).any() or (away_diff < 0).any():
        return 'invalid_sequence'

    # 2. Final score
    last_row = group.iloc[-1]

    home_final = last_row['home_point']
    away_final = last_row['away_point']

    winner_score = max(home_final, away_final)
    score_diff = abs(home_final - away_final)

    if winner_score < 7:
        return 'invalid_final_score'

    if score_diff < 2:
        return 'invalid_difference'

    return 'valid'

tie_break_validation = (
    tie_break_data
    .groupby(['match_id', 'set_id', 'game_id'])
    .apply(validate_tie_break)
)

tie_break_validation.value_counts()

C:\Users\ASMA\AppData\Local\Temp\ipykernel_4400\1470305586.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(validate_tie_break)


valid                  3165
invalid_final_score       4
invalid_sequence          1
Name: count, dtype: int64

# detect invalid tie break game data

In [40]:
invalid_tie_breaks = (
    tie_break_validation[tie_break_validation != 'valid']
)
invalid_games = (
    tie_break_data
    .set_index(['match_id', 'set_id', 'game_id'])
    .loc[invalid_tie_breaks.index]
    .reset_index()
)

invalid_games['match_id'].unique()

array([11999211, 12088042, 12100263, 12149669, 12159545])

In [41]:
def fix_score_sequence(group):
    group = group.sort_values('point_id').copy()

    for i in range(1, len(group)):
        
        if group.iloc[i]['home_point'] < group.iloc[i-1]['home_point']:
            group.iloc[i, group.columns.get_loc('home_point')] = \
                group.iloc[i-1]['home_point']

        if group.iloc[i]['away_point'] < group.iloc[i-1]['away_point']:
            group.iloc[i, group.columns.get_loc('away_point')] = \
                group.iloc[i-1]['away_point']

    return group


# Create valid & invalid tie break game datafram

In [42]:
def clean_tie_breaks(tie_break_data):

    valid_tie_break_games = []
    invalid_tie_break_games = []

    # بررسی هر تای‌بریک
    for game_id, group in tie_break_data.groupby(
        ['match_id', 'set_id', 'game_id']
    ):

        # مرحله اول: اعتبارسنجی
        result = validate_tie_break(group)

        # اگر مشکل توالی داشت، اصلاح کن
        if result == 'invalid_sequence':
            group = fix_score_sequence(group)

            # بعد از اصلاح دوباره اعتبارسنجی کن
            result = validate_tie_break(group)

        # اگر معتبر بود
        if result == 'valid':
            valid_tie_break_games.append(group)

        # اگر امتیاز نهایی نامعتبر بود
        elif result == 'invalid_final_score':
            invalid_tie_break_games.append(group)

    # ساخت DataFrameهای نهایی
    tie_break_valid_data = pd.concat(
        valid_tie_break_games,
        ignore_index=True
    )

    invalid_tie_break_data = pd.concat(
        invalid_tie_break_games,
        ignore_index=True
    )

    
    return tie_break_valid_data, invalid_tie_break_data

tie_break_valid_data, invalid_tie_break_data= (
    clean_tie_breaks(tie_break_data)
)

print ("shape of tie break vaild data :" , tie_break_valid_data.shape)
print ("shape of tie break invalid data :" , invalid_tie_break_data.shape)

shape of tie break vaild data : (39136, 14)
shape of tie break invalid data : (38, 14)


# Create normal game dataframe

In [43]:
normal_game_data = pbp_data[ pbp_data['game_type'] == 'normal_game'].copy()
normal_game_data.shape

(1215570, 14)

# validate normal game data

In [44]:
def validate_normal_game(group):
    group = group.sort_values('point_id').copy()

    valid_scores = {'0', '15', '30', '40', 'A'}

    # 1. مقادیر امتیاز باید معتبر باشند
    if not group['home_point'].isin(valid_scores).all():
        return 'invalid_score'

    if not group['away_point'].isin(valid_scores).all():
        return 'invalid_score'

    scores = list(
        zip(group['home_point'], group['away_point'])
    )

    score_order = {
        '0': 0,
        '15': 1,
        '30': 2,
        '40': 3
    }

    # 2. بررسی توالی امتیازها
    for i in range(1, len(scores)):

        previous = scores[i - 1]
        current = scores[i]

        # Advantage
        if previous in [('A', '40'), ('40', 'A')]:

            if current == ('40', '40'):
                continue

            return 'invalid_sequence'

        # Deuce
        if previous == ('40', '40'):

            if current in [
                ('A', '40'),
                ('40', 'A'),
                ('40', '40')
            ]:
                continue

            return 'invalid_sequence'

        # قبل از Deuce
        previous_home = previous[0]
        previous_away = previous[1]
        current_home = current[0]
        current_away = current[1]

        # A نباید قبل از Deuce ظاهر شود
        if current_home == 'A' or current_away == 'A':
            return 'invalid_sequence'

        home_change = (
            score_order[current_home] -
            score_order[previous_home]
        )

        away_change = (
            score_order[current_away] -
            score_order[previous_away]
        )

        if home_change < 0 or away_change < 0:
            return 'invalid_sequence'

        # در هر Point فقط یکی از بازیکنان امتیاز می‌گیرد
        if home_change > 0 and away_change > 0:
            return 'invalid_sequence'

        # حداقل یکی باید تغییر کرده باشد
        if home_change == 0 and away_change == 0:
            return 'invalid_sequence'

    # 3. بررسی نتیجه نهایی Game
    last_home = group.iloc[-1]['home_point']
    last_away = group.iloc[-1]['away_point']

    # حالت‌های برد قبل از Deuce
    valid_winning_scores = [
        ('40', '0'),
        ('40', '15'),
        ('40', '30'),
        ('0', '40'),
        ('15', '40'),
        ('30', '40')
    ]

    if (last_home, last_away) in valid_winning_scores:
        return 'valid'

    # برد با Advantage
    if (last_home, last_away) in [
        ('A', '40'),
        ('40', 'A')
    ]:
        return 'valid'

    # اگر به اینجا رسیدیم، Game با نتیجه معتبر تمام نشده
    return 'invalid_final_score'

normal_game_validation = (
    normal_game_data
    .groupby(['match_id', 'set_id', 'game_id'])
    .apply(validate_normal_game)
)

normal_game_validation.value_counts()

C:\Users\ASMA\AppData\Local\Temp\ipykernel_4400\934336850.py:111: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(validate_normal_game)


valid                  224709
invalid_final_score       264
invalid_sequence           82
Name: count, dtype: int64

# Create valid & invalid normal game data dataframe

In [45]:
valid_normal_keys = normal_game_validation[
    normal_game_validation == 'valid'
].index

invalid_normal_keys = normal_game_validation[
    normal_game_validation != 'valid'
].index

normal_game_valid = (
    normal_game_data
    .set_index(['match_id', 'set_id', 'game_id'])
    .loc[valid_normal_keys]
    .reset_index()
)

normal_game_invalid = (
    normal_game_data
    .set_index(['match_id', 'set_id', 'game_id'])
    .loc[invalid_normal_keys]
    .reset_index()
) 
print ('shape valid normal game data' , normal_game_valid.shape )
print ('shape invalid normal game data:', normal_game_invalid.shape)

shape valid normal game data (1213280, 14)
shape invalid normal game data: (2290, 14)


# Create final valid & invalid pbp data (sum of normal and tie break data)

In [46]:
final_pbp = pd.concat(
    [
        normal_game_valid,
        tie_break_valid_data
    ],
    ignore_index=True
)
final_pbp['home_point'] = final_pbp['home_point'].astype('string')
final_pbp['away_point'] = final_pbp['away_point'].astype('string')

invalid_pbp = pd.concat(
    [
        normal_game_invalid,
        invalid_tie_break_data
    ],
    ignore_index=True
)
invalid_pbp['home_point'] = invalid_pbp['home_point'].astype('string')
invalid_pbp['away_point'] = invalid_pbp['away_point'].astype('string')

print ('shape valid pbp data' , final_pbp.shape )
print ('shape invalid pbp data:', invalid_pbp.shape)

shape valid pbp data (1252416, 14)
shape invalid pbp data: (2328, 14)


# Save outputs

In [47]:
valid_output_path = r'C:\Users\ASMA\Desktop\code\Tennis_project\data\final_data\final_pbp.parquet'

final_pbp.to_parquet(
    valid_output_path,
    index=False
)
print ('the final pbp data is saved')

invalid_output_path = r'C:\Users\ASMA\Desktop\code\Tennis_project\data\final_data\invalid_pbp.parquet'

invalid_pbp.to_parquet(
    invalid_output_path,
    index=False
)

print('The invalid PBP data is saved')

the final pbp data is saved
The invalid PBP data is saved
